# Terraform Pipeline Executor - Explicit Phases Demo

Ce notebook démontre l'utilisation du **PipelineExecutor** qui structure la génération Terraform en 4 phases explicites:

1. **📋 PLANNING**: Analyse des requirements + recherche knowledge base
2. **🔧 GENERATION**: Génération du code + validation syntaxe
3. **🔍 CODE REVIEW**: Vérifications sécurité + best practices
4. **✅ VALIDATION**: Terraform plan + rapport final

Chaque phase produit un output structuré avec des checks explicites (✅/❌/⚠️).

## Étape 1: Imports et Configuration

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import agent classes
import sys
sys.path.insert(0, str(Path.cwd().parent))

from terraform_agent import Config, PromptManager, KnowledgeBase, PipelineExecutor

print("✓ All imports loaded successfully")

## Étape 2: Configuration du Logging

In [ ]:
import logging

# Configure logging to see pipeline execution
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

# Optional: Set specific loggers to DEBUG for more detail
logging.getLogger('terraform_agent').setLevel(logging.INFO)

print("✓ Logging configured")

## Étape 3: Initialisation des Composants

Initialise tous les composants nécessaires:
- **Config**: Chemins et modèles
- **PromptManager**: Templates système et utilisateur
- **KnowledgeBase**: ChromaDB avec best practices
- **PipelineExecutor**: Orchestrateur avec phases explicites

In [ ]:
# Get project root
project_root = Path.cwd().parent

# Initialize configuration
config = Config(base_dir=project_root)
print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Work Dir: {config.WORK_DIR}")

# Initialize prompt manager
prompts = PromptManager(config)
print("\n✓ Prompts loaded")

# Initialize knowledge base
knowledge_base = KnowledgeBase(config)

# Initialize pipeline executor
pipeline = PipelineExecutor(config, prompts, knowledge_base)

print("\n✓ Pipeline executor ready")

## Étape 4: Exécution du Pipeline

Le pipeline s'exécute en 4 phases avec output structuré:

### Phase 1: PLANNING 📋
- Analyse du prompt utilisateur
- Recherche dans la knowledge base
- Création du plan d'exécution

### Phase 2: GENERATION 🔧
- Génération du code Terraform
- `terraform init`
- `terraform validate`

### Phase 3: CODE REVIEW 🔍
- Vérifications sécurité (UBLA, encryption, public access)
- Best practices compliance (module structure, variables, outputs)
- Scoring (security score, BP score)

### Phase 4: VALIDATION ✅
- `terraform plan`
- Résumé de toutes les phases
- Verdict final (SUCCESS/WARNING/FAILED)

In [ ]:
# Load user prompt
prompt_filename = "user_prompts/1-bucket.md"
user_prompt = (Path.cwd().parent / prompt_filename).read_text()

# Execute pipeline (all 4 phases)
result = pipeline.run(user_prompt=user_prompt)

## Étape 5: Rapport d'Exécution

Récupère un rapport détaillé de l'exécution avec:
- Log de chaque phase (status, durée, détails)
- Durée totale
- Timestamp
- Répertoire de travail

In [ ]:
# Get execution report
import json

report = pipeline.get_execution_report()
print("\n📊 Execution Report:")
print(json.dumps(report, indent=2))

## Étape 6: Vérification des Fichiers Générés

In [ ]:
# List generated Terraform files
work_dir = config.WORK_DIR
tf_files = sorted(work_dir.rglob("*.tf"))

print(f"\n📁 Generated files ({len(tf_files)} total):")
for tf_file in tf_files:
    relative_path = tf_file.relative_to(work_dir)
    file_size = tf_file.stat().st_size
    print(f"   {relative_path} ({file_size} bytes)")

## Comparaison: Pipeline vs Generator Standard

### TerraformGenerator (standard)
```
🚀 Starting agent...
✅ Done
```

### PipelineExecutor (avec phases)
```
📋 PHASE 1: PLANNING
  ✅ Requirements analyzed
  ✅ Knowledge base searched (3 chunks)
  
🔧 PHASE 2: GENERATION
  ✅ terraform init
  ✅ terraform validate
  
🔍 PHASE 3: CODE REVIEW
  ✅ UBLA enabled
  ✅ Public access prevention
  Security score: 80%
  
✅ PHASE 4: VALIDATION
  ✅ terraform plan
  ✅ PIPELINE SUCCEEDED
```

Le **PipelineExecutor** rend le workflow transparent et auditable.